# Imports

In [ ]:
#running on google colab
on_google_colab = False

In [ ]:
#google colab installs
if on_google_colab:
    !pip3 install pydicom

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import pydicom
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
#seed used for random
seed = 42
torch.manual_seed(seed)

In [ ]:
#device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [ ]:
#colab settings
if on_google_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/upenn_measurement_project/model_directory')

# Measurement data

In [ ]:
#read csv
measurement_file = "08242025.csv"

df = pd.read_csv(f"../data/measurements/{measurement_file}")

print(df.columns)

#drop unnecessary columns
df.drop(columns = ["Student Name", "Notes", "Date Completed", "Unnamed: 24", "Unnamed: 25", "Unnamed: 26", "Unnamed: 27"], inplace = True)

print(df.columns)

print(len(df.columns.values))

In [ ]:
#get rid of images with na (only temporary for convenience)
df = df.dropna()

# Image data

In [ ]:
#image directory
img_dir = "../data/box_images"

#train test split
train_test_split = 0.8
val_test_split = 0.5

#batch size
train_batch_size = 64
val_batch_size = 8
test_batch_size = 8

#pixels per mm conversion (known)
pix_per_mm = 2400 / 408

#training scale factor
img_scale_factor = 0.1

#image size (pix)
img_width = int(2400 * img_scale_factor)
img_height = int(1920 * img_scale_factor)

In [ ]:
#data augmentation settings
##rotation
rot_int = 30
rot_st = -60
rot_en = 60

##cropping
crop_int = 0.2
crop_st = 0
crop_en = 0.4

##x offset
x_int = 50
x_st = -50
x_en = 0

##y offset
y_int = 50
y_st = -50
y_en = 0

In [ ]:
#class for each data point used for keypoint detection
class DataPoint:
    def __init__(self, img_dir, img_file, img_flip, df, img_width, img_height, rot = 0, crop = 0, transx = 0, transy = 0):
        self.img_path = img_dir + '/' + img_file
        self.img_flip = img_flip
        self.img_width = img_width
        self.img_height = img_height
        self.measurements = torch.from_numpy(df.loc[df['ID'] == img_file].drop(columns = 'ID').values.astype(np.float32)).reshape(-1)

        self.rot = rot
        self.crop = crop
        self.transx = int(transx)
        self.transy = int(transy)
        self.scale = 1

        #additionally requried scale for source img rotation
        abscos = np.abs(np.cos(self.rot * np.pi / 180))
        abssin = np.abs(np.sin(self.rot * np.pi / 180))
        self.scale *= min(1 / (abscos + self.img_height / self.img_width * abssin), 1 / (self.img_width / self.img_height * abssin + abscos))

    #return augmented image and measurements
    def get_image(self):
        ds = pydicom.dcmread(self.img_path)
        img = ds.pixel_array

        img = torch.from_numpy(img).float()
        img = (img - img.min()) / (img.max() - img.min() + 1e-5)
        img = img.unsqueeze(0).unsqueeze(0)

        #flip image if needed
        if self.img_flip:
            img = TF.hflip(img)

        #crop image
        _, _, orig_height, orig_width = img.shape
        crop_mask = torch.zeros_like(img)
        crop_mask[:, :, int(orig_height * self.crop):, int(orig_width * self.crop):] = 1
        img = img * crop_mask

        #downscale and pad
        new_height = int(self.img_height * self.scale)
        new_width = int(self.img_width * self.scale)
        img = F.interpolate(img, size = (new_height, new_width), mode = 'bilinear', align_corners = False)

        pad_height = self.img_height - new_height
        pad_width = self.img_width - new_width
        pad_u = pad_height // 2
        pad_d = pad_height - pad_u
        pad_l = pad_width // 2
        pad_r = pad_width - pad_l
        img = F.pad(img, (pad_l, pad_r, pad_u, pad_d), mode = 'constant')

        #rotate image
        img = TF.rotate(img, angle = self.rot, interpolation = TF.InterpolationMode.BILINEAR)

        #offset image
        def offset_image(img, x, y, fill = 0):
            res = torch.full_like(img, fill)

            stx = max(x, 0)
            enx = min(x + self.img_width, self.img_width)
            sty = max(y, 0)
            eny = min(y + self.img_height, self.img_height)

            res[:, :, sty:eny, stx:enx] = img[:, :, max(0, -y):min(self.img_height - y, self.img_height), max(0, -x):min(self.img_width - x, self.img_width)]

            return res

        img = offset_image(img, self.transx, self.transy)

        #augmented measurements
        aug_measurements = self.measurements.clone()
        if self.img_flip: #left side of image, right set of measurements
            aug_measurements = aug_measurements[10:]
        else: #right side of image, left set of measurements
            aug_measurements = aug_measurements[:10]
        aug_measurements[:9] *= self.scale

        return img.squeeze(0), aug_measurements, torch.tensor(self.scale).float()

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, data_points):
        self.data_points = data_points

    def __len__(self):
        return len(self.data_points)

    def __getitem__(self, idx):
        return self.data_points[idx].get_image()

In [ ]:
#all images in directory
all_img_in_dir = [i for i in os.listdir(img_dir) if i.lower().endswith("dcm")]

#check all images in df are loaded (else throw error)
for i in df['ID'].values:
    if i not in all_img_in_dir:
        raise SystemError(f'Directory is missing images: {i}')

#load only images in the df
all_img = [i for i in all_img_in_dir if i in df['ID'].values]
print(f'Images in use: {len(all_img)}/{len(all_img_in_dir)}')

#train and val sizes
train_sz = int(train_test_split * len(all_img))
val_sz = int(val_test_split * (len(all_img) - train_sz))

#split train, val, test sets
all_idx = torch.randperm(len(all_img))
train_img = [all_img[i] for i in all_idx[:train_sz]]
val_img = [all_img[i] for i in all_idx[train_sz:train_sz + val_sz]]
test_img = [all_img[i] for i in all_idx[train_sz + val_sz:]]

In [ ]:
#augmented training data in DataPoint class
train_data_points = []
for img_file in train_img:
    for img_flip in range(2):
        for rot in np.arange(rot_st, rot_en + 1e-8, rot_int):
            for crop in np.arange(crop_st, crop_en + 1e-8, crop_int):
                for x in np.arange(x_st, x_en + 1e-8, x_int):
                    for y in np.arange(y_st, y_en + 1e-8, y_int):
                        train_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height, rot, crop, x, y))
print(f'Augmented training set size: {len(train_data_points)}')

#validation data in DataPoint class (no augmentation transformations applied)
val_data_points = []
for img_file in val_img:
    for img_flip in range(2):
        val_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height))
print(f'Validation set size: {len(val_data_points)}')

#testing data in DataPoint class (no augmentation transformations applied)
test_data_points = []
for img_file in test_img:
    for img_flip in range(2):
        test_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height))
print(f'Testing set size: {len(test_data_points)}')

In [ ]:
#training dataset and data loader
train_set = ImageDataset(train_data_points)
train_loader = DataLoader(train_set, batch_size = train_batch_size, shuffle = True)

#validation dataset and data loader
val_set = ImageDataset(val_data_points)
val_loader = DataLoader(val_set, batch_size = val_batch_size, shuffle = False)

#testing dataset and data loader
test_set = ImageDataset(test_data_points)
test_loader = DataLoader(test_set, batch_size = test_batch_size, shuffle = False)

In [ ]:
#quick test
testcnt = 1
testchannelcnt = 1
for imgs, ys, aug_scales in train_loader:
    for i in range(testchannelcnt):
        plt.imshow(imgs[0][i], cmap = 'gray')
        plt.axis('off')
        plt.show()

    print(ys[0])
    print(aug_scales[0])

    testcnt -= 1
    if testcnt == 0:
        break


# Direct heatmap model

## Heatmap/coordinate/measurement conversion functions

In [ ]:
#heatmaps to true points on image
def heatmaps_to_model_points(heatmaps, img_width, img_height):
    device = heatmaps.device

    B, K, H, W = heatmaps.shape
    heatmaps = heatmaps.view(B, K, -1)
    softmax_heatmaps = F.softmax(heatmaps, dim=-1)
    softmax_heatmaps = softmax_heatmaps.view(B, K, H, W)

    xs = torch.linspace(-1, 1, W, device = device)
    ys = torch.linspace(-1, 1, H, device = device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing = 'ij')
    grid_x = grid_x.unsqueeze(0).unsqueeze(0)
    grid_y = grid_y.unsqueeze(0).unsqueeze(0)

    expected_x = (softmax_heatmaps * grid_x).sum(dim= [-2, -1])
    expected_y = (softmax_heatmaps * grid_y).sum(dim= [-2, -1])

    points = torch.stack([expected_x, expected_y], dim = 2)

    points = (points + 1) / 2
    points[:, :, 0] = points[:, :, 0] * img_width
    points[:, :, 1] = points[:, :, 1] * img_height

    return points

In [ ]:
def model_points_to_coord(model_points):
    coord = model_points.clone()

    #center on A and remove A
    coord = coord - coord[:, 0:1, :]
    coord = coord[:, 1:, :]

    #rotate to relative coordinates (point order: D, F, G, H, W, X, Y, Z)
    wz = torch.norm(coord[:, 4, :] - coord[:, 7, :], dim = -1)
    sin = -(coord[:, 4, 1] - coord[:, 7, 1]) / wz
    cos = -(coord[:, 4, 0] - coord[:, 7, 0]) / wz

    rot_mat = torch.stack((
        torch.stack((cos, -sin), dim=-1),
        torch.stack((sin,  cos), dim=-1),
    ), dim=-2)
    coord = coord @ rot_mat

    #invert y
    coord[:, :, 1] = -coord[:, :, 1]

    #get AB
    slope = coord[:, 0, 1] / (coord[:, 0, 0] + 1e-6)
    b_x = (coord[:, 3, 0] / (slope + 1e-6) + coord[:, 3, 1]) / (slope + 1 / (slope + 1e-6) + 1e-6)
    b_y = slope * b_x
    ab = torch.sqrt(b_x ** 2 + b_y ** 2)

    return coord, ab

In [ ]:
#turn measurements into points
def measurements_to_coord(measurements, ab, pix_per_mm, img_scale_factor):
    measurements = measurements.clone()
    measurements[:, :9] *= pix_per_mm * img_scale_factor

    ang = (270 - measurements[:, 9]) * torch.pi / 180
    cos = torch.cos(ang)
    sin = torch.sin(ang)

    b_x = -cos * ab
    b_y = -sin * ab
    c_x = measurements[:, 4]
    c_y = -measurements[:, 5]

    all_x_y = [None for _ in range(8)]
    all_x_y[0] = torch.stack([cos * measurements[:, 3] / 2, sin * measurements[:, 3] / 2], dim = -1) #D
    all_x_y[2] = torch.stack([all_x_y[0][:, 0] - cos * measurements[:, 8], all_x_y[0][:, 1] - sin * measurements[:, 8]], dim = -1) #G
    all_x_y[1] = torch.stack([all_x_y[2][:, 0] + cos * measurements[:, 7], all_x_y[2][:, 1] + sin * measurements[:, 7]], dim = -1) #F
    all_x_y[3] = torch.stack([b_x + sin * measurements[:, 6] / 2, b_y - cos * measurements[:, 6] / 2], dim = -1) #H
    all_x_y[4] = torch.stack([c_x - measurements[:, 2] / 2, c_y], dim = -1) #W
    all_x_y[7] = torch.stack([c_x + measurements[:, 2] / 2, c_y], dim = -1) #Z
    all_x_y[5] = torch.stack([all_x_y[4][:, 0] + measurements[:, 0], c_y], dim = -1) #X
    all_x_y[6] = torch.stack([all_x_y[7][:, 0] - measurements[:, 1], c_y], dim = -1) #Y

    coord = torch.stack(all_x_y, dim = 1)

    return coord

In [ ]:
#turn points into measurements
def coord_to_measurements(coord, pix_per_mm, img_scale_factor):
    eps = 1e-6

    slope = coord[:, 0, 1] / (coord[:, 0, 0] + eps)
    b_x = (coord[:, 3, 0] / (slope + eps) + coord[:, 3, 1]) / (slope + 1 / (slope + eps) + eps)
    b_y = slope * b_x

    measurements = [None for _ in range(10)]
    measurements[0] = coord[:, 5, 0] - coord[:, 4, 0]
    measurements[1] = coord[:, 7, 0] - coord[:, 6, 0]
    measurements[2] = coord[:, 7, 0] - coord[:, 4, 0]
    measurements[3] = 2 * torch.norm(coord[:, 0], dim = 1)
    measurements[4] = (coord[:, 4, 0] + coord[:, 7, 0]) / 2
    measurements[5] = -coord[:, 4, 1]
    measurements[6] = 2 * torch.sqrt((coord[:, 3, 0] - b_x) ** 2 + (coord[:, 3, 1] - b_y) ** 2)
    measurements[7] = torch.norm(coord[:, 1] - coord[:, 2], dim = 1)
    measurements[8] = torch.norm(coord[:, 0] - coord[:, 2], dim = 1)
    measurements[9] = torch.arctan(-slope) * 180 / torch.pi + 90

    measurements = torch.stack(measurements, dim = -1).clone()
    measurements[:, :9] /= pix_per_mm * img_scale_factor

    return measurements


## Architecture

In [ ]:
#keypoint detection encoder
class HeatmapModel(nn.Module):
    def __init__(self, keypoint_cnt, img_height, img_width, channel_cnt = [1, 32, 64, 128, 256]):
        super().__init__()

        self.img_height = img_height
        self.img_width = img_width

        self.conv1 = nn.Sequential(
            nn.Conv2d(channel_cnt[0], channel_cnt[1], kernel_size = 7, stride = 1, padding = 3), #192, 240
            nn.ReLU(),
            nn.Conv2d(channel_cnt[1], channel_cnt[1], kernel_size = 3, stride = 1, padding = 1), #192, 240
            nn.ReLU()
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(channel_cnt[1], channel_cnt[2], kernel_size = 3, stride = 2, padding = 1), #96, 120
            nn.ReLU(),
            nn.Conv2d(channel_cnt[2], channel_cnt[2], kernel_size = 3, stride = 1, padding = 1), #96, 120
            nn.ReLU()
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(channel_cnt[2], channel_cnt[3], kernel_size = 3, stride = 2, padding = 1), #48, 60
            nn.ReLU(),
            nn.Conv2d(channel_cnt[3], channel_cnt[3], kernel_size = 3, stride = 1, padding = 1), #48, 60
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(channel_cnt[3], channel_cnt[4], kernel_size = 3, stride = 2, padding = 1), #24, 30
            nn.ReLU(),
            nn.Conv2d(channel_cnt[4], channel_cnt[4], kernel_size = 3, stride = 1, padding = 1), #24, 30
            nn.ReLU()
        )

        self.conv5 = nn.Conv2d(channel_cnt[4], keypoint_cnt, kernel_size = 1, stride = 1, padding = 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        return x

## Training

In [ ]:
#model
model = HeatmapModel(9, img_height, img_width) #number of poi + 1 (because A)

#number of epochs
epoch_cnt = 100

#learning rate
learning_rate = 1e-3

#early stopping conditions
best_val_loss = float('inf')
early_stop_lim = 3
early_stop_cnt = 0

In [ ]:
#loss function
lossfn = lambda ypred, yvals, scales : torch.sum(torch.sum(torch.norm(ypred - yvals, dim = 2), dim = 1) / scales)

#loss history (validation)
loss_hist = []

In [ ]:
#optimizer
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:
for epoch in range(epoch_cnt):
    model.to(device)
    model.train()

    total_loss = 0

    for images, yvals, scales in train_loader:
        images = images.to(device)
        yvals = yvals.to(device)
        scales = scales.to(device)

        heatmaps = model(images)
        points = heatmaps_to_model_points(heatmaps, img_width, img_height)
        pred_coord, ab = model_points_to_coord(points)
        real_coord = measurements_to_coord(yvals, ab, pix_per_mm, img_scale_factor)

        loss = lossfn(pred_coord, real_coord, scales)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch: {epoch}\n')
    print(f'Loss (in sample): {total_loss / len(train_set)}\n')

    total_loss = 0
    total_percent_err = torch.zeros(10).to(device)

    model.eval()
    with torch.no_grad():
        for images, yvals, scales in val_loader:
            images = images.to(device)
            yvals = yvals.to(device)
            scales = scales.to(device)

            heatmaps = model(images)
            points = heatmaps_to_model_points(heatmaps, img_width, img_height)
            pred_coord, ab = model_points_to_coord(points)
            real_coord = measurements_to_coord(yvals, ab, pix_per_mm, img_scale_factor)
            pred_measurements = coord_to_measurements(pred_coord, pix_per_mm, img_scale_factor)

            loss = lossfn(pred_coord, real_coord, scales)
            total_loss += loss.item()

            total_percent_err += torch.sum(torch.abs(yvals - pred_measurements) / yvals, dim = 0)

        print(f'Loss (validation): {total_loss / len(val_set)} (best {best_val_loss / len(val_set)})\n')
        print(f'Percent error for each measurement (validation):')
        for i in total_percent_err:
            print(f'{(i.item() / len(val_set)):.4f}', end = ' ')
        print()

        print('===\n')

    loss_hist.append(total_loss / len(val_set))

    if total_loss > best_val_loss:
        early_stop_cnt += 1
    else:
        early_stop_cnt = 0
        best_val_loss = total_loss
        torch.save(model.state_dict(), "best_model.pth")

    if early_stop_cnt >= early_stop_lim:
        print('EARLY STOPPED')
        break

plt.plot(loss_hist)
plt.show()

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))

## Testing

In [ ]:
model.to(device)
model.eval()

total_loss = 0
total_percent_err = torch.zeros(10).to(device)

with torch.no_grad():
    for images, yvals, scales in test_loader:
        images = images.to(device)
        yvals = yvals.to(device)
        scales = scales.to(device)

        heatmaps = model(images)
        points = heatmaps_to_model_points(heatmaps, img_width, img_height)
        
        pred_coord, ab = model_points_to_coord(points)
        real_coord = measurements_to_coord(yvals, ab, pix_per_mm, img_scale_factor)
        pred_measurements = coord_to_measurements(pred_coord, pix_per_mm, img_scale_factor)

        loss = lossfn(pred_coord, real_coord, scales)
        total_loss += loss.item()

        total_percent_err += torch.sum(torch.abs(yvals - pred_measurements) / yvals, dim = 0)

        plt.figure(figsize = (10, 3))

        plt.subplot(1, 2, 1)
        plt.imshow(images[0, 0].detach().cpu(), cmap = 'gray')
        plt.scatter(points[0, :, 0].detach().cpu(), points[0, :, 1].detach().cpu(), c = 'r')

        plt.subplot(1, 2, 2)
        plt.scatter(real_coord[0, :, 0].detach().cpu(), real_coord[0, :, 1].detach().cpu(), c = 'b')
        plt.scatter(pred_coord[0, :, 0].detach().cpu(), pred_coord[0, :, 1].detach().cpu(), c = 'r')
        
        plt.show()

        print(pred_coord[0])

    print(f'Loss (out sample): {total_loss / len(test_set)}\n')
    print(f'Percent error for each measurement (out sample):')
    for i in total_percent_err:
        print(f'{(i.item() / len(test_set)):.4f}', end = ' ')
    print()

## Debugging

In [ ]:
model.to(device)
model.eval()

total_loss = 0
total_percent_err = torch.zeros(10).to(device)

with torch.no_grad():
    for images, yvals, scales in train_loader:
        images = images.to(device)
        yvals = yvals.to(device)
        scales = scales.to(device)

        SANA = torch.full_like(images, fill_value = 0.5)
        print(scales[0])

        heatmaps = model(SANA)
        points = heatmaps_to_model_points(heatmaps, img_width, img_height)
        
        pred_coord, ab = model_points_to_coord(points)
        real_coord = measurements_to_coord(yvals, ab, pix_per_mm, img_scale_factor)
        pred_measurements = coord_to_measurements(pred_coord, pix_per_mm, img_scale_factor)

        loss = lossfn(pred_coord, real_coord, scales)
        total_loss += loss.item()

        total_percent_err += torch.sum(torch.abs(yvals - pred_measurements) / yvals, dim = 0)

        plt.figure(figsize = (10, 3))

        plt.subplot(1, 2, 1)
        plt.imshow(SANA[0, 0].detach().cpu(), cmap = 'gray', vmin=0, vmax=1)
        plt.scatter(points[0, :, 0].detach().cpu(), points[0, :, 1].detach().cpu(), c = 'r')

        plt.subplot(1, 2, 2)
        plt.scatter(real_coord[0, :, 0].detach().cpu(), real_coord[0, :, 1].detach().cpu(), c = 'b')
        plt.scatter(pred_coord[0, :, 0].detach().cpu(), pred_coord[0, :, 1].detach().cpu(), c = 'r')
        
        plt.show()

        print(pred_coord[0])

    print(f'Loss (out sample): {total_loss / len(train_set)}\n')
    print(f'Percent error for each measurement (out sample):')
    for i in total_percent_err:
        print(f'{(i.item() / len(train_set)):.4f}', end = ' ')
    print()

In [ ]:
torch.set_printoptions(sci_mode=False, linewidth=120, precision=5)
for name, p in model.named_parameters():
    print(f"\n=== {name} ===\n{p.data}")


# Other

## Save model

In [ ]:
# torch.save(model.state_dict(), "MODEL_NAME.pth")

## Load model

In [ ]:
model.load_state_dict(torch.load("model_saves/first_direct_heatmap.pth", map_location = torch.device("cpu")))
        